In [0]:
spark.sql("SHOW TABLES IN workspace.nba_silver").show(200, truncate=False)

In [0]:
spark.table("workspace.nba_silver.fact_odds_h2h_history").count()
spark.table("workspace.nba_silver.fact_odds_h2h_latest").count()

In [0]:
# =====================================================================================
# Notebook: 02_silver/04_fact_game_odds_h2h_latest.py
# Project : nba-datalakehouse
# Purpose : Join The Odds API H2H (latest) to enriched games and create a Silver table
#           for "latest odds per game" analysis.
#
# Inputs (Silver):
#   - workspace.nba_silver.fact_game_enriched
#   - workspace.nba_silver.fact_odds_h2h_latest
#
# Output (Silver):
#   - workspace.nba_silver.fact_game_odds_h2h_latest
#
# Notes:
# - We join on (normalized home/away team names) + (time tolerance between commence and start)
# - We keep odds grain per bookmaker/outcome (still useful for line shopping)
# - We avoid IndexError if join is empty
# =====================================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------
# 0) Table names
# -----------------------------
FACT_GAME_ENRICHED = "workspace.nba_silver.fact_game_enriched"
ODDS_H2H_LATEST = "workspace.nba_silver.fact_odds_h2h_latest"
OUT_TABLE = "workspace.nba_silver.fact_game_odds_h2h_latest"

print("FACT_GAME_ENRICHED =", FACT_GAME_ENRICHED)
print("ODDS_H2H_LATEST    =", ODDS_H2H_LATEST)
print("OUT_TABLE          =", OUT_TABLE)

# -----------------------------
# 1) Load inputs
# -----------------------------
games = spark.table(FACT_GAME_ENRICHED)
odds = spark.table(ODDS_H2H_LATEST)

print("games rows =", games.count())
print("odds  rows =", odds.count())

# -----------------------------
# 2) Normalize team names for a reliable join
# -----------------------------
def norm_team_name(c):
    return F.lower(
        F.regexp_replace(
            F.regexp_replace(F.trim(c), r"[^a-zA-Z0-9\s]", ""),
            r"\s+",
            " "
        )
    )

games_j = (
    games
    .select(
        "game_id_bdl",
        "season_start_year",
        "game_date_utc",
        "start_time_utc",
        "status",
        "home_team_name",
        "visitor_team_name",
        "source_dt"
    )
    .withColumn("home_team_norm", norm_team_name(F.col("home_team_name")))
    .withColumn("away_team_norm", norm_team_name(F.col("visitor_team_name")))
)

odds_j = (
    odds
    .select(
        "source_dt",
        "ingested_at",
        "event_id",
        "sport_key",
        "sport_title",
        "commence_time_utc",
        "home_team_name",
        "away_team_name",
        "bookmaker_key",
        "bookmaker_title",
        "bookmaker_last_update_utc",
        "market_key",
        "market_last_update_utc",
        "outcome_name",
        "price_decimal",
        "source_system"
    )
    .withColumn("home_team_norm", norm_team_name(F.col("home_team_name")))
    .withColumn("away_team_norm", norm_team_name(F.col("away_team_name")))
)

# -----------------------------
# 3) Join odds -> games with time tolerance
# -----------------------------
# Why tolerance?
# - Odds commence_time_utc is when the book says the game starts
# - BallDontLie start_time_utc can be rounded/normalized
# - Sometimes there are timezone/format differences
#
# 8 hours is intentionally wide for robustness while you’re learning.
TOLERANCE_SECONDS = 8 * 3600

joined = (
    odds_j.alias("o")
    .join(
        games_j.alias("g"),
        on=[
            F.col("o.home_team_norm") == F.col("g.home_team_norm"),
            F.col("o.away_team_norm") == F.col("g.away_team_norm"),
            F.abs(F.unix_timestamp(F.col("o.commence_time_utc")) - F.unix_timestamp(F.col("g.start_time_utc"))) <= F.lit(TOLERANCE_SECONDS),
        ],
        how="inner"
    )
)

fact_game_odds = (
    joined
    .select(
        F.col("g.source_dt").alias("source_dt"),
        F.col("g.game_id_bdl").alias("game_id_bdl"),
        F.col("g.season_start_year").alias("season_start_year"),
        F.col("g.game_date_utc").alias("game_date_utc"),
        F.col("g.start_time_utc").alias("start_time_utc"),
        F.col("g.status").alias("game_status"),
        F.col("g.home_team_name").alias("home_team_name"),
        F.col("g.visitor_team_name").alias("away_team_name"),

        F.col("o.event_id").alias("event_id"),
        F.col("o.sport_key").alias("sport_key"),
        F.col("o.sport_title").alias("sport_title"),
        F.col("o.commence_time_utc").alias("commence_time_utc"),

        F.col("o.bookmaker_key").alias("bookmaker_key"),
        F.col("o.bookmaker_title").alias("bookmaker_title"),
        F.col("o.bookmaker_last_update_utc").alias("bookmaker_last_update_utc"),

        F.col("o.market_key").alias("market_key"),
        F.col("o.market_last_update_utc").alias("market_last_update_utc"),

        F.col("o.outcome_name").alias("outcome_name"),
        F.col("o.price_decimal").alias("price_decimal"),

        F.col("o.ingested_at").alias("odds_ingested_at"),
        F.col("o.source_system").alias("source_system"),
    )
    .withColumn(
        "seconds_between_commence_and_start",
        F.unix_timestamp(F.col("start_time_utc")) - F.unix_timestamp(F.col("commence_time_utc"))
    )
    .withColumn(
        "seconds_from_odds_update_to_start",
        F.unix_timestamp(F.col("start_time_utc")) - F.unix_timestamp(F.col("market_last_update_utc"))
    )
)

# -----------------------------
# 4) Deduplicate safely
# -----------------------------
# Keep the newest odds snapshot per unique key for each game.
w = Window.partitionBy(
    "game_id_bdl", "bookmaker_key", "market_key", "outcome_name", "market_last_update_utc"
).orderBy(F.col("odds_ingested_at").desc())

fact_game_odds = (
    fact_game_odds
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# -----------------------------
# 5) Write output table
# -----------------------------
spark.sql(f"DROP TABLE IF EXISTS {OUT_TABLE}")

(
    fact_game_odds.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("source_dt")
    .saveAsTable(OUT_TABLE)
)

out_rows = fact_game_odds.count()
print("Wrote:", OUT_TABLE, "rows=", out_rows)

# -----------------------------
# 6) Validations (won't crash if join is empty)
# -----------------------------
print("\n=== Partitions (source_dt) ===")
spark.table(OUT_TABLE).groupBy("source_dt").count().orderBy(F.col("source_dt").desc()).show(20, truncate=False)

print("\n=== Time difference sanity (seconds) ===")
spark.table(OUT_TABLE).select(
    F.max("seconds_between_commence_and_start").alias("max_diff"),
    F.percentile_approx("seconds_between_commence_and_start", 0.5).alias("p50_diff"),
    F.percentile_approx("seconds_between_commence_and_start", 0.9).alias("p90_diff"),
).show(truncate=False)

print("\n=== Sample rows ===")
display(
    spark.table(OUT_TABLE)
    .orderBy(F.col("market_last_update_utc").desc())
    .select(
        "game_id_bdl","home_team_name","away_team_name",
        "start_time_utc","commence_time_utc",
        "bookmaker_key","outcome_name","price_decimal",
        "market_last_update_utc",
        "seconds_between_commence_and_start",
        "seconds_from_odds_update_to_start"
    )
    .limit(200)
)

if out_rows == 0:
    print("\nNo joined rows produced.")
    print("Most common reasons:")
    print("1) Odds are for future games outside your games dataset window")
    print("2) Team naming mismatch (rare, but possible)")
    print("3) commence_time_utc/start_time_utc mismatch > tolerance (increase tolerance to 24h temporarily)")